# Aula 02: Grafos de Computação, Autograd e a Primeira MLP (Previsão de Churn em SaaS)

Bem-vindo(a) à segunda aula do nosso curso de PyTorch! Na aula anterior, aprendemos a matemática do Perceptron, as funções de ativação e a manipulação de tensores. 

Nesta aula, daremos o grande salto: entenderemos como o PyTorch calcula derivadas automaticamente via **Autograd** e como estruturar o **ciclo completo de treinamento** de redes neurais. Primeiro, aplicaremos o treinamento automático ao Perceptron da Creatina (retomando o código movido da Aula 1). Em seguida, expandiremos para a nossa primeira **Rede Neural Multicamadas (MLP)** aplicada à predição de **Churn de Clientes** em uma empresa de SaaS de assinatura.

## Objetivos desta aula:
1. Compreender o funcionamento dos **Grafos de Computação** e o motor de diferenciação automática **`Autograd`**.
2. Treinar um modelo em PyTorch usando a API orientada a objetos (`nn.Module`), funções de perda (`nn.BCELoss`) e otimizadores (`optim.SGD`).
3. Analisar o impacto da **Taxa de Aprendizado (Learning Rate)** na convergência do treinamento.
4. Construir uma arquitetura de **MLP (Multi-Layer Perceptron)** com camadas ocultas para classificar Churn de clientes SaaS.

--- 

## 1. Grafos de Computação e Autograd no PyTorch

O PyTorch constrói dinamicamente um **Grafo Direcionado Acíclico (DAG)** durante a passagem direta (*forward pass*). 

### Conceitos Chave:
* **Entrada ($x$)**: Dado fixo da amostra do dataset (não necessita de gradientes).
* **Pesos ($w$) e Viés ($b$)**: Parâmetros treináveis marcados com `requires_grad=True`.
* **Palpite ($\hat{y}$ ou `y_hat`)**: Estimativa gerada pela rede neural.
* **Meta Real ($y$ ou `y_real`)**: O valor correto/esperado do rótulo.
* **Norma do Erro ($L$ ou `loss`)**: Métrica de distância entre a estimativa e a meta real.

Quando chamamos `loss.backward()`, o PyTorch percorre o grafo em sentido inverso (*backpropagation*) aplicando a Regra da Cadeia para encontrar os gradientes exatos dos parâmetros treináveis (`w.grad` e `b.grad`).

In [ ]:
import torch

# 1. Entrada de dados fixa (dado do cliente) e Meta Real desejada (y_real)
x = torch.tensor(2.0)                       # Dado fixo do dataset (sem gradientes)
y_real = torch.tensor(1.0)                  # Meta real desejada

# 2. Parâmetros Treináveis da Rede (requires_grad=True)
w = torch.tensor(3.0, requires_grad=True)   # Peso treinável
b = torch.tensor(1.0, requires_grad=True)   # Viés treinável

# 3. Forward Pass: Calculando a estimativa y_hat
z = w * x + b                               # Soma ponderada: z = 3*2 + 1 = 7.0
y_hat = z                                   # Palpite estimado da rede: y_hat = 7.0

# 4. Norma do Erro / Função de Perda (Loss L)
loss = (y_hat - y_real) ** 2                # L = (7.0 - 1.0)^2 = 36.0

print(f"Palpite do Modelo (y_hat): {y_hat.item()}")
print(f"Meta Real Desejada (y_real): {y_real.item()}")
print(f"Norma do Erro / Loss (L): {loss.item()}")

# 5. Retropropagação (Backpropagation via Autograd)
loss.backward()

# Gradientes calculados pela Regra da Cadeia apenas para os parâmetros treináveis:
# dL/dw = dL/dy_hat * dy_hat/dz * dz/dw = 2*(y_hat - y_real) * 1 * x = 2*(7 - 1) * 2 = 24.0
# dL/db = dL/dy_hat * dy_hat/dz * dz/db = 2*(y_hat - y_real) * 1 * 1 = 2*(7 - 1) * 1 = 12.0
print(f"\nGradiente do Peso (w.grad = dL/dw): {w.grad.item()}")
print(f"Gradiente do Viés (b.grad = dL/db): {b.grad.item()}")

--- 

## 2. Criando e Treinando o Perceptron no PyTorch (`nn.Module`)

*(Código de Treinamento movido da Aula 01)*

Agora que entendemos como o Autograd calcula gradientes, vamos construir o neurônio artificial para os dados da Creatina usando a API padrão do PyTorch. O próprio framework gerenciará os parâmetros e atualizará os pesos usando **Backpropagation** e **Gradiente Descendente**.

In [ ]:
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Recriando o Dataset dos Atletas
dados = {
    "Creatina_g": [5, 1, 3, 0, 8, 2, 6, 4, 10, 2, 7, 3, 5, 1, 9, 0, 4, 2, 8, 3],
    "Treino_h": [8, 2, 6, 4, 7, 3, 5, 2, 9, 5, 4, 2, 6, 6, 3, 8, 8, 5, 2, 3],
    "Performance_Real": [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]
}
df = pd.DataFrame(dados)
X = torch.tensor(df[['Creatina_g', 'Treino_h']].values, dtype=torch.float32)
Y_real = torch.tensor(df['Performance_Real'].values, dtype=torch.float32).unsqueeze(1)

# Definindo a classe do Perceptron herdando de nn.Module
class Perceptron(nn.Module):
    def __init__(self):
        super(Perceptron, self).__init__()
        self.linear = nn.Linear(in_features=2, out_features=1)
        
    def forward(self, x):
        return torch.sigmoid(self.linear(x))

# Inicializando modelo, função de perda e otimizador
modelo = Perceptron()
criterio = nn.BCELoss()
otimizador = optim.SGD(modelo.parameters(), lr=0.1)

# Loop de Treinamento por 1000 épocas
epocas = 1000
historico_perda = []

for epoca in range(epocas):
    # 1. Zerar gradientes acumulados
    otimizador.zero_grad()
    
    # 2. Forward Pass
    predicoes = modelo(X)
    
    # 3. Calcular Erro (Loss)
    perda = criterio(predicoes, Y_real)
    historico_perda.append(perda.item())
    
    # 4. Backpropagation (Autograd)
    perda.backward()
    
    # 5. Atualização dos Pesos
    otimizador.step()
    
    if (epoca + 1) % 200 == 0:
        print(f"Época {epoca+1:04d}/{epocas} - Loss: {perda.item():.4f}")

In [ ]:
# Visualizando a Curva de Aprendizado e a Fronteira de Decisão Otimizada
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(historico_perda, color='blue', linewidth=2)
ax1.set_xlabel('Época')
ax1.set_ylabel('Erro Médio (BCELoss)')
ax1.set_title('Curva de Aprendizado (Convergência)')
ax1.grid(True, alpha=0.3)

pesos = modelo.linear.weight.data[0]
bias = modelo.linear.bias.data[0]
x_vals = np.linspace(0, 10, 100)
y_vals = -(pesos[0].item() * x_vals + bias.item()) / pesos[1].item()

ax2.scatter(df[df['Performance_Real']==1]['Creatina_g'], df[df['Performance_Real']==1]['Treino_h'], color='green', marker='^', s=80, label='Alta Perf.')
ax2.scatter(df[df['Performance_Real']==0]['Creatina_g'], df[df['Performance_Real']==0]['Treino_h'], color='red', marker='o', s=80, label='Reg. Perf.')
ax2.plot(x_vals, y_vals, color='blue', linestyle='--', linewidth=2, label='Fronteira Otimizada')
ax2.set_xlim(-0.5, 10.5)
ax2.set_ylim(-0.5, 10.5)
ax2.set_title('Fronteira de Decisão Aprendida')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

--- 

## 3. Laboratório de Hiperparâmetros: Análise da Taxa de Aprendizado (Learning Rate)

*(Movido da Aula 01)*

A taxa de aprendizado (`lr`) determina o tamanho do passo no sentido do gradiente descendente. Vamos testar 4 taxas de aprendizado e analisar o impacto na estabilidade da perda.

In [ ]:
taxas_de_aprendizado = [10.0, 0.1, 0.01, 0.0001]
plt.figure(figsize=(10, 5))

for lr in taxas_de_aprendizado:
    modelo_lab = Perceptron()
    criterio_lab = nn.BCELoss()
    otimizador_lab = optim.SGD(modelo_lab.parameters(), lr=lr)
    historico = []
    
    for epoca in range(500):
        otimizador_lab.zero_grad()
        pred = modelo_lab(X)
        loss = criterio_lab(pred, Y_real)
        loss.backward()
        otimizador_lab.step()
        historico.append(loss.item())
        
    plt.plot(historico, label=f'lr = {lr}')

plt.xlabel('Época')
plt.ylabel('Loss (BCELoss)')
plt.title('Impacto da Taxa de Aprendizado (Learning Rate) na Convergência')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

--- 

## 4. Primeira MLP (Multi-Layer Perceptron) para Previsão de Churn em SaaS

Agora aplicaremos nossos conhecimentos em um problema complexo e real de negócios: prever o **Churn (cancelamento de assinatura)** dos clientes de um SaaS de tecnologia.

### Variáveis do SaaS:
* **Tempo de Contrato (meses)**
* **Faturamento Mensal ($)**
* **Chamados de Suporte Abertos**
* **Dias sem Acessar a Plataforma**

In [ ]:
# Sintetizando um dataset realista de SaaS Telco Churn
np.random.seed(42)
n_amostras = 500

tempo_contrato = np.random.randint(1, 48, n_amostras)        # 1 a 48 meses
faturamento = np.random.uniform(29.90, 299.90, n_amostras)   # valor mensal
chamados_suporte = np.random.randint(0, 10, n_amostras)      # quantidade
dias_inativo = np.random.randint(0, 30, n_amostras)          # dias

# Regra com ruído para gerar Churn (1 = cancelou, 0 = ativo)
score_churn = (chamados_suporte * 0.4) + (dias_inativo * 0.3) - (tempo_contrato * 0.1) + np.random.normal(0, 1, n_amostras)
churn = (score_churn > 2.0).astype(int)

df_saas = pd.DataFrame({
    'Tempo_Contrato': tempo_contrato,
    'Faturamento': faturamento,
    'Chamados_Suporte': chamados_suporte,
    'Dias_Inativo': dias_inativo,
    'Churn': churn
})

print("Primeiras linhas do dataset de Churn SaaS:")
print(df_saas.head())

### 4.1 Construindo a Arquitetura da MLP em PyTorch (`nn.Module`)

Nossa rede terá:
* **Camada de Entrada**: 4 características de entrada.
* **Camada Oculta**: 8 neurônios com ativação **ReLU**.
* **Camada de Saída**: 1 neurônio com ativação **Sigmoid**.

In [ ]:
# Normalizando as entradas e convertendo para tensores
X_raw = df_saas[['Tempo_Contrato', 'Faturamento', 'Chamados_Suporte', 'Dias_Inativo']].values
X_norm = (X_raw - X_raw.mean(axis=0)) / X_raw.std(axis=0)

X_tensor = torch.tensor(X_norm, dtype=torch.float32)
Y_tensor = torch.tensor(df_saas['Churn'].values, dtype=torch.float32).unsqueeze(1)

# Definindo a arquitetura da MLP
class MLPChurn(nn.Module):
    def __init__(self):
        super(MLPChurn, self).__init__()
        self.camada_oculta = nn.Linear(in_features=4, out_features=8)
        self.relu = nn.ReLU()
        self.camada_saida = nn.Linear(in_features=8, out_features=1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        h = self.relu(self.camada_oculta(x))
        out = self.sigmoid(self.camada_saida(h))
        return out

modelo_mlp = MLPChurn()
criterio_mlp = nn.BCELoss()
otimizador_mlp = optim.Adam(modelo_mlp.parameters(), lr=0.01)

# Loop de Treinamento
epocas_mlp = 300
historico_loss_mlp = []

for epoca in range(epocas_mlp):
    otimizador_mlp.zero_grad()
    pred = modelo_mlp(X_tensor)
    loss = criterio_mlp(pred, Y_tensor)
    loss.backward()
    otimizador_mlp.step()
    historico_loss_mlp.append(loss.item())
    
    if (epoca + 1) % 50 == 0:
        print(f"Época {epoca+1:03d}/{epocas_mlp} - Loss: {loss.item():.4f}")

In [ ]:
# Avaliando a acurácia final da MLP
with torch.no_grad():
    pred_prob = modelo_mlp(X_tensor)
    pred_classes = (pred_prob >= 0.5).float()
    acuracia_final = (pred_classes == Y_tensor).float().mean() * 100

print(f"Acurácia Final da MLP no Dataset de Churn: {acuracia_final.item():.2f}%")

plt.figure(figsize=(7, 4))
plt.plot(historico_loss_mlp, color='green', linewidth=2)
plt.xlabel('Época')
plt.ylabel('Erro Médio (BCELoss)')
plt.title('Convergência do Treinamento da MLP (SaaS Churn)')
plt.grid(True, alpha=0.3)
plt.show()

--- 

## 5. Resumo e Próximos Passos

Nesta aula aprendemos:
1. Como o motor de diferenciação **Autograd** calcula derivadas automaticamente.
2. O ciclo completo de 5 passos para treinar qualquer modelo no PyTorch.
3. O comportamento do **Learning Rate** durante a otimização.
4. Como construir uma **MLP (Multi-Layer Perceptron)** com camadas ocultas para resolver um problema real de Churn em SaaS.

👉 **Na Aula 03**: Estudaremos o ciclo de treinamento profissional em conjuntos de treino/validação e a modelagem multiclasse tabular!